# Tarea 1 — LIM y CONC

Fragmenta el corpus, etiqueta fragmentos por función retórica y produce tres salidas:
el dataset de entrenamiento con heurísticas (Dataset A), el mismo dataset con los
patrones de encabezado neutralizados para evitar data leakage (Dataset B), y una
muestra para validación humana inter-anotador.

## 1. Instalación

In [ ]:
!pip install -q pandas pyarrow tqdm

## 2. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración

In [ ]:
NOMBRE              = "Camilo"
ETIQUETAS_ASIGNADAS = ["LIM", "CONC"]

INPUT_PATH        = "/content/drive/MyDrive/camilo_corpus.parquet"
OUTPUT_A_PATH     = "/content/drive/MyDrive/camilo_dataset_A.parquet"
OUTPUT_B_PATH     = "/content/drive/MyDrive/camilo_dataset_B.parquet"
OUTPUT_VALID_PATH = "/content/drive/MyDrive/camilo_validacion_humana.csv"

MAX_POR_ETIQUETA = 2000
N_VALIDACION     = 200
SEED             = 42
MIN_PALABRAS     = 250
MAX_PALABRAS     = 1000

TARGET_WORDS_PER_FRAGMENT = 450
PARAGRAPH_OVERLAP         = 1
MAX_FRAGMENTS_PER_DOC     = 2
N_LINEAS_ENCABEZADO       = 5

print(f"Miembro:             {NOMBRE}")
print(f"Etiquetas asignadas: {ETIQUETAS_ASIGNADAS}")
print(f"Cuota entrenamiento: {MAX_POR_ETIQUETA} por etiqueta")
print(f"Cuota validacion:    {N_VALIDACION} por etiqueta")

## 4. Patrones

In [ ]:
import re

# Patrones de encabezado: se buscan en las primeras N_LINEAS_ENCABEZADO no vacias
HEADING_PATTERNS = {
    "INTRO": [
        r"introducci[oó]n",
        r"planteamiento del problema",
        r"motivaci[oó]n",
        r"objetivo(s)? (del|de este)",
    ],
    "BACK": [
        r"trabajo(s)? relacionado(s)?",
        r"estado del arte",
        r"antecedentes",
        r"marco te[oó]rico",
    ],
    "METH": [
        r"metodolog[ií]a",
        r"m[eé]todo(s)?",
        r"dise[nñ]o experimental",
        r"materiales y m[eé]todos",
    ],
    "RES": [
        r"resultado(s)?",
        r"experimento(s)?",
        r"evaluaci[oó]n emp[ií]rica",
    ],
    "DISC": [
        r"discusi[oó]n(?:es)?",
        r"an[aá]lisis(?: y discusi[oó]n)?",
        r"discusi[oó]n de resultados",
        r"interpretaci[oó]n de resultados",
        r"implicaciones",
    ],
    "CONTR": [
        r"contribuciones?",
        r"aportes?",
        r"principales? contribuciones?",
        r"contribuci[oó]n(es)? del trabajo",
        r"aportaci[oó]n(es)?",
    ],
    "LIM": [
        r"limitaci[oó]n(es)?",
        r"restricciones?",
        r"trabajo futuro",
        r"l[ií]neas? futuras?",
        r"alcance y limitaciones",
        r"limitaciones del estudio",
    ],
    "CONC": [
        r"conclusi[oó]n(es)?",
        r"hallazgo(s)? principal(es)?",
        r"en resumen",
        r"a modo de conclusi[oó]n",
        r"resumen final",
        r"consideraciones? finales?",
    ],
}

LEXICAL_PATTERNS = {
    "LIM": [
        r"no considera",
        r"fuera del alcance",
        r"como limitaci[oó]n",
        r"entre las limitaciones",
        r"una limitaci[oó]n",
        r"sin embargo este enfoque",
        r"como trabajo futuro",
        r"futuras investigaciones",
        r"l[ií]neas de investigaci[oó]n futuras",
        r"queda fuera del alcance",
        r"no fue posible",
        r"requiere mayor investigaci[oó]n",
    ],
    "CONC": [
        r"en conclusi[oó]n",
        r"concluimos",
        r"este trabajo ha demostrado",
        r"los resultados muestran que",
        r"finalmente[,]? este trabajo",
        r"en s[ií]ntesis",
        r"a lo largo de este trabajo",
        r"hemos demostrado",
        r"este estudio concluye",
        r"los hallazgos principales",
        r"podemos concluir",
    ],
}

NEGATIVE_PATTERNS = {
    "LIM": [
        r"referencias",
        r"bibliograf[ií]a",
        r"agradecimientos",
        r"metodolog[ií]a",
    ],
    "CONC": [
        r"referencias",
        r"bibliograf[ií]a",
        r"agradecimientos",
        r"anexo",
        r"ap[eé]ndice",
    ],
}

_HEADING_COMPILADOS = {
    etiqueta: [re.compile(p, re.IGNORECASE) for p in patrones]
    for etiqueta, patrones in HEADING_PATTERNS.items()
}

print(f"Etiquetas definidas: {list(HEADING_PATTERNS.keys())}")
print(f"Etiquetas activas:   {ETIQUETAS_ASIGNADAS}")

## 5. Imports

In [ ]:
import json
import math
import os
import random
import unicodedata
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 160)

## 6. Limpieza de texto

In [ ]:
def strip_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")


def normalize_whitespace(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def remove_non_content_lines(text: str) -> str:
    cleaned = []
    for line in text.split("\n"):
        raw = line.strip()
        if not raw:
            cleaned.append("")
            continue
        if re.fullmatch(r"\d+", raw):
            continue
        if re.fullmatch(r"(?:figura|figure|tabla|table)\s*\d+[.:\-]?", raw.lower()):
            continue
        if re.fullmatch(r"https?://\S+", raw.lower()):
            continue
        if re.fullmatch(r"doi\s*[:.?]?\s*\S+", raw.lower()):
            continue
        cleaned.append(raw)
    return "\n".join(cleaned)


def remove_reference_section(text: str) -> str:
    patterns = [r"\nreferencias\n", r"\nbibliograf[ií]a\n", r"\nreferences\n"]
    lowered = text.lower()
    for pat in patterns:
        match = re.search(pat, lowered)
        if match and match.start() > len(text) * 0.55:
            return text[:match.start()].strip()
    return text


def clean_document_text(text: str) -> str:
    text = normalize_whitespace(text)
    text = remove_non_content_lines(text)
    text = remove_reference_section(text)
    text = normalize_whitespace(text)
    return text


def word_count(text: str) -> int:
    return len(text.split())

## 7. Fragmentación

In [ ]:
@dataclass
class Fragment:
    doc_id: str
    filename: str
    fragment_index: int
    text: str
    word_count: int
    start_paragraph: int
    end_paragraph: int
    relative_start: float
    relative_end: float
    nearest_heading: str


HEADING_LINE_REGEX = re.compile(
    r"^(?:\d+(?:\.\d+)*)?\s*([A-ZÁÉÍÓÚÑ][A-ZÁÉÍÓÚÑa-záéíóúñ\-\s]{2,80})$"
)


def is_heading_line(line: str) -> bool:
    line = line.strip()
    if not line or len(line) > 90:
        return False
    if line.count(".") > 2:
        return False
    if line.endswith(":"):
        return True
    return bool(HEADING_LINE_REGEX.match(line))


def split_into_paragraphs(text: str) -> List[str]:
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]


def build_fragments_for_document(doc_id: str, filename: str, text: str) -> List[Fragment]:
    paragraphs = split_into_paragraphs(text)
    if not paragraphs:
        return []

    fragments = []
    total_paragraphs = len(paragraphs)
    current_heading = ""
    i = 0
    fragment_idx = 0

    while i < total_paragraphs:
        start = i
        buffer = []
        n_words = 0
        local_heading = current_heading

        while i < total_paragraphs:
            paragraph = paragraphs[i]
            if is_heading_line(paragraph):
                current_heading = paragraph
                if n_words >= MIN_PALABRAS:
                    break
                local_heading = current_heading
                i += 1
                continue
            p_words = word_count(paragraph)
            if p_words == 0:
                i += 1
                continue
            if n_words + p_words > MAX_PALABRAS and n_words >= MIN_PALABRAS:
                break
            buffer.append(paragraph)
            n_words += p_words
            i += 1
            if n_words >= TARGET_WORDS_PER_FRAGMENT and n_words >= MIN_PALABRAS:
                break

        while i < total_paragraphs and n_words < MIN_PALABRAS:
            paragraph = paragraphs[i]
            if is_heading_line(paragraph):
                current_heading = paragraph
                i += 1
                continue
            p_words = word_count(paragraph)
            if n_words + p_words > MAX_PALABRAS:
                break
            buffer.append(paragraph)
            n_words += p_words
            i += 1

        if buffer and MIN_PALABRAS <= n_words <= MAX_PALABRAS:
            end = i - 1
            fragments.append(Fragment(
                doc_id=str(doc_id),
                filename=str(filename),
                fragment_index=fragment_idx,
                text="\n\n".join(buffer).strip(),
                word_count=n_words,
                start_paragraph=start,
                end_paragraph=end,
                relative_start=start / max(total_paragraphs, 1),
                relative_end=(end + 1) / max(total_paragraphs, 1),
                nearest_heading=local_heading,
            ))
            fragment_idx += 1
            i = max(start + 1, i - PARAGRAPH_OVERLAP)
        else:
            i = max(i, start + 1)

    return fragments


def etiquetar_fragmento(fragment: Fragment) -> str:
    lineas = [l for l in fragment.text.splitlines() if l.strip()]
    encabezado = " ".join(lineas[:N_LINEAS_ENCABEZADO])

    texto_busqueda = f"{fragment.nearest_heading} {encabezado}"

    for etiqueta in ETIQUETAS_ASIGNADAS:
        for patron in _HEADING_COMPILADOS[etiqueta]:
            if patron.search(texto_busqueda):
                return etiqueta
    return "UNKNOWN"

## 8. Cargar corpus

In [ ]:
df_corpus = pd.read_parquet(INPUT_PATH)
df_corpus = df_corpus.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Documentos: {len(df_corpus):,}")
print(f"Columnas:   {df_corpus.columns.tolist()}")
df_corpus.head(2)

## 9. Fragmentar y etiquetar

In [ ]:
registros = []

for row in tqdm(df_corpus.itertuples(index=False), total=len(df_corpus), desc="Fragmentando"):
    texto_limpio = clean_document_text(row.texto)
    if word_count(texto_limpio) < MIN_PALABRAS:
        continue

    fragmentos = build_fragments_for_document(row.doc_id, row.filename, texto_limpio)

    for frag in fragmentos:
        etiqueta = etiquetar_fragmento(frag)
        registros.append({
            "doc_id":         frag.doc_id,
            "fragmento_id":   f"{frag.doc_id}_{frag.fragment_index:04d}",
            "texto":          frag.text,
            "etiqueta_auto":  etiqueta,
            "num_palabras":   frag.word_count,
            "relative_start": round(frag.relative_start, 4),
            "nearest_heading": frag.nearest_heading,
            "asignado_a":     NOMBRE,
        })

df_fragmentos = pd.DataFrame(registros)

dist = df_fragmentos["etiqueta_auto"].value_counts()
n_unknown = dist.get("UNKNOWN", 0)
pct_unknown = 100 * n_unknown / max(len(df_fragmentos), 1)

print(f"Total fragmentos generados: {len(df_fragmentos):,}")
print()
print("Distribucion de etiquetas:")
print(dist.to_string())
print()
print(f"UNKNOWN: {n_unknown:,} ({pct_unknown:.1f}%) — indica cuantos fragmentos no tienen encabezado reconocible")

## 10. Filtrado por etiqueta

In [ ]:
df_filtrado = df_fragmentos[
    df_fragmentos["etiqueta_auto"].isin(ETIQUETAS_ASIGNADAS)
].copy().reset_index(drop=True)

print(f"Fragmentos disponibles tras filtrar UNKNOWN: {len(df_filtrado):,}")
print()

cuota_total = MAX_POR_ETIQUETA + N_VALIDACION
for etiqueta in ETIQUETAS_ASIGNADAS:
    n = (df_filtrado["etiqueta_auto"] == etiqueta).sum()
    suficiente = "OK" if n >= cuota_total else f"ADVERTENCIA: faltan {cuota_total - n} fragmentos"
    print(f"  {etiqueta}: {n:,} disponibles (cuota: {cuota_total}) — {suficiente}")

## 11. Separar validacion y entrenamiento

In [ ]:
# validacion se extrae primero para garantizar que no aparezca en train
df_validacion = (
    df_filtrado
    .groupby("etiqueta_auto", group_keys=False)
    .apply(lambda g: g.sample(n=min(N_VALIDACION, len(g)), random_state=SEED))
    .reset_index(drop=True)
)

ids_val = set(df_validacion["fragmento_id"])

# entrenamiento se samplea del pool restante
df_train = (
    df_filtrado[~df_filtrado["fragmento_id"].isin(ids_val)]
    .groupby("etiqueta_auto", group_keys=False)
    .apply(lambda g: g.sample(n=min(MAX_POR_ETIQUETA, len(g)), random_state=SEED))
    .reset_index(drop=True)
)

interseccion = ids_val & set(df_train["fragmento_id"])
assert len(interseccion) == 0, (
    f"Data leakage detectado: {len(interseccion)} fragmento(s) aparecen "
    f"tanto en validacion como en entrenamiento: {list(interseccion)[:5]}"
)

print("Entrenamiento:")
print(df_train["etiqueta_auto"].value_counts().to_string())
print()
print("Validacion:")
print(df_validacion["etiqueta_auto"].value_counts().to_string())
print()
print("Interseccion train/validacion: 0 (OK)")

## 12. Dataset A — guardar

In [ ]:
df_train.to_parquet(OUTPUT_A_PATH, index=False)
mb_a = os.path.getsize(OUTPUT_A_PATH) / (1024 ** 2)

print(f"Dataset A guardado en: {OUTPUT_A_PATH}")
print(f"Tamanio: {mb_a:.1f} MB")
print()
print("Fragmentos por etiqueta:")
for etiqueta in ETIQUETAS_ASIGNADAS:
    n = (df_train["etiqueta_auto"] == etiqueta).sum()
    print(f"  {etiqueta}: {n:,}")

## 13. Dataset B — desacople de patrones

In [ ]:
TOKEN_NEUTRO = "[SECCION]"


def desacoplar_patrones(texto: str, etiqueta: str) -> tuple:
    if etiqueta not in _HEADING_COMPILADOS:
        return texto, False

    patrones = _HEADING_COMPILADOS[etiqueta]
    lineas = texto.splitlines()
    lineas_no_vacias_vistas = 0
    modificado = False
    resultado = []
    primera_ocurrencia_reemplazada = False

    for linea in lineas:
        if linea.strip():
            lineas_no_vacias_vistas += 1

        # Zona de encabezado: titulo y subtitulo (primeras N lineas)
        if lineas_no_vacias_vistas <= N_LINEAS_ENCABEZADO and linea.strip():
            if any(p.search(linea) for p in patrones):
                resultado.append(TOKEN_NEUTRO)
                modificado = True
                continue

        # Cuerpo: solo la primera ocurrencia del patron
        elif not primera_ocurrencia_reemplazada:
            nueva_linea = linea
            for patron in patrones:
                if patron.search(linea):
                    nueva_linea = patron.sub(TOKEN_NEUTRO, linea, count=1)
                    primera_ocurrencia_reemplazada = True
                    modificado = True
                    break
            resultado.append(nueva_linea)
            continue

        resultado.append(linea)

    return "\n".join(resultado), modificado


# Aplicar desacople sobre df_train
df_train_b = df_train.copy()

textos_b = df_train_b.apply(
    lambda r: desacoplar_patrones(r["texto"], r["etiqueta_auto"]),
    axis=1
)

df_train_b["texto"]       = textos_b.apply(lambda t: t[0])
df_train_b["desacoplado"] = textos_b.apply(lambda t: t[1])

# Reporte de fragmentos modificados por etiqueta
print("Fragmentos modificados por etiqueta:")
for etiqueta in ETIQUETAS_ASIGNADAS:
    subset = df_train_b[df_train_b["etiqueta_auto"] == etiqueta]
    n_mod = subset["desacoplado"].sum()
    pct   = 100 * n_mod / max(len(subset), 1)
    print(f"  {etiqueta}: {n_mod:,} / {len(subset):,} ({pct:.1f}%)")

# Guardar Dataset B
df_train_b.to_parquet(OUTPUT_B_PATH, index=False)
mb_b = os.path.getsize(OUTPUT_B_PATH) / (1024 ** 2)
print(f"\nDataset B guardado en: {OUTPUT_B_PATH}")
print(f"Tamanio: {mb_b:.1f} MB")

# Mostrar 2 ejemplos lado a lado
print("\n" + "="*70)
print("EJEMPLOS DE DESACOPLE (original vs desacoplado)")
print("="*70)
ejemplos = df_train_b[df_train_b["desacoplado"] == True].head(2)
for i, (_, row) in enumerate(ejemplos.iterrows(), 1):
    original = df_train[df_train["fragmento_id"] == row["fragmento_id"]]["texto"].values[0]
    print(f"\nEjemplo {i} — {row['etiqueta_auto']}")
    print("  ORIGINAL:    ", original[:200].replace("\n", " "))
    print("  DESACOPLADO: ", row["texto"][:200].replace("\n", " "))


## 14. Planilla de validacion humana

In [ ]:
planilla = pd.DataFrame({
    "doc_id":          df_validacion["doc_id"],
    "fragmento_id":    df_validacion["fragmento_id"],
    "texto_corto":     df_validacion["texto"].str[:300],
    "etiqueta_auto":   df_validacion["etiqueta_auto"],
    "etiqueta_humano": "",
    "anotador":        NOMBRE,
    "notas":           "",
})

planilla.to_csv(OUTPUT_VALID_PATH, index=False)

print(f"Planilla guardada en: {OUTPUT_VALID_PATH}")
print(f"Total filas: {len(planilla):,}")
print()
print("Distribucion por etiqueta:")
print(planilla["etiqueta_auto"].value_counts().to_string())

## 15. Resumen final

In [ ]:
def n_mod_por_etiqueta(df, etiqueta):
    subset = df[df["etiqueta_auto"] == etiqueta]
    n_mod = subset["desacoplado"].sum()
    pct   = 100 * n_mod / max(len(subset), 1)
    return len(subset), n_mod, pct

print("RESUMEN FINAL")
print("=" * 55)
print()
print("Dataset A — entrenamiento con heuristicas:")
for etiqueta in ETIQUETAS_ASIGNADAS:
    n = (df_train["etiqueta_auto"] == etiqueta).sum()
    print(f"  {etiqueta}: {n:,} fragmentos")

print()
print("Dataset B — entrenamiento desacoplado:")
for etiqueta in ETIQUETAS_ASIGNADAS:
    n, n_mod, pct = n_mod_por_etiqueta(df_train_b, etiqueta)
    print(f"  {etiqueta}: {n:,} fragmentos ({n_mod:,} modificados, {pct:.1f}%)")

print()
print("Validacion humana:")
for etiqueta in ETIQUETAS_ASIGNADAS:
    n = (df_validacion["etiqueta_auto"] == etiqueta).sum()
    print(f"  {etiqueta}: {n:,} fragmentos")

print()
print("Archivos generados:")
print(f"  {OUTPUT_A_PATH}")
print(f"  {OUTPUT_B_PATH}")
print(f"  {OUTPUT_VALID_PATH}")